## OpenAI com skill prompts

Esta secao usa os prompts da skill `ecossistema-agentes-prompts` para orquestrador, especialistas e avaliador.

In [2]:
import os
import re
from typing import Dict, List, Optional, Tuple
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# Caminho base das skills (pode ser absoluto ou relativo ao notebook)
SKILLS_DIR = "mnt/skills"
SKILL_FILE = "SKILL.md"

def _parse_frontmatter(content: str) -> Tuple[Dict, str]:
    """Extrai frontmatter YAML (entre ---) e retorna (metadados, resto do conteúdo)."""
    meta = {}
    body = content
    match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)", content, re.DOTALL)
    if match:
        fm, body = match.group(1), match.group(2)
        for line in fm.split("\n"):
            if ":" in line:
                key, val = line.split(":", 1)
                meta[key.strip()] = val.strip().strip("'\"").replace("\n", " ").strip()
    return meta, body.strip()

def load_skills(base_dir: str) -> List[Dict]:
    """
    Carrega todas as skills a partir de base_dir.
    Cada subpasta contendo SKILL.md vira uma skill.
    Retorna lista de dicts: id, name, description, content (texto completo).
    """
    skills = []
    base = os.path.abspath(base_dir)
    if not os.path.isdir(base):
        return skills
    for entry in sorted(os.listdir(base)):
        path = os.path.join(base, entry)
        skill_file = os.path.join(path, SKILL_FILE)
        if os.path.isdir(path) and os.path.isfile(skill_file):
            with open(skill_file, "r", encoding="utf-8") as f:
                raw = f.read()
            meta, body = _parse_frontmatter(raw)
            skill_id = entry
            name = meta.get("name", skill_id)
            description = meta.get("description", "")
            skills.append({
                "id": skill_id,
                "name": name,
                "description": description,
                "content": raw,
                "body": body,
            })
    return skills

def get_skill_by_id(skills: List[Dict], skill_id: str) -> Optional[Dict]:
    """Retorna a skill com o id dado ou None."""
    for s in skills:
        if s["id"] == skill_id:
            return s
    return None

# Carrega todas as skills
skills_loaded = load_skills(SKILLS_DIR)
# print(f"Skills carregadas: {len(skills_loaded)}")
# for s in skills_loaded:
#     print(f"  - {s['id']}: {s.get('name', s['id'])}")
#     print(s['content'])

In [21]:
def chat_with_skill(
    client: OpenAI,
    skill_id: str,
    user_message: str,
    model: str,
    skills_list: Optional[List[Dict]] = None,
) -> str:
    """
    Envia uma mensagem ao modelo OpenAI usando o conteúdo da skill como system prompt.
    Se skills_list for None, usa skills_loaded (carregadas na célula anterior).
    """
    skills = skills_list if skills_list is not None else skills_loaded
    skill = get_skill_by_id(skills, skill_id)
    if not skill:
        return f"Erro: skill '{skill_id}' não encontrada."
    system = skill["content"]
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_message},
        ],
    )
    return resp.choices[0].message.content or ""

# Exemplo:
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
# model = os.environ.get("MODELO_DEFAULT")
# resposta = chat_with_skill(client, "agente-tecnico", "Como implementar um RAG com embeddings?", model)
# print(resposta)

In [22]:
import json

def extrair_json(texto: str) -> Optional[Dict]:
    """Extrai JSON do texto retornado pelo modelo (pode vir em bloco ```json ... ```)."""
    if not texto or not texto.strip():
        return None
    s = texto.strip()
    # Remove bloco markdown de código
    if "```" in s:
        match = re.search(r"```(?:json)?\s*([\s\S]*?)```", s)
        if match:
            s = match.group(1).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return None

def invocar_agente_maestro(
    client: OpenAI,
    skill_id: str,
    payload_maestro: Dict,
    model: str,
    skills_list: Optional[List[Dict]] = None,
) -> str:
    """
    Invoca um agente em modo Maestro; o modelo deve retornar JSON no Formato de Retorno da skill.
    Retorna o texto bruto da resposta (parse com extrair_json depois).
    """
    skills = skills_list if skills_list is not None else skills_loaded
    skill = get_skill_by_id(skills, skill_id)
    if not skill:
        return ""
    system = skill["content"]
    user = (
        json.dumps(payload_maestro, ensure_ascii=False)
        + "\n\nVocê está sendo invocado pelo Maestro. Responda APENAS com um único JSON válido no Formato de Retorno da skill "
        "(agente_id, agente_nome, pode_responder, justificativa_viabilidade, resposta, scores, limitacoes_da_resposta, aspectos_para_outros_agentes). "
        "Sem texto antes ou depois."
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        response_format={"type": "json_object"},
    )
    return (resp.choices[0].message.content or "").strip()

In [ ]:
def executar_fluxo_maestro(
    client: OpenAI,
    pergunta: str,
    model: str = None,
    skills_list: Optional[List[Dict]] = None,
    agentes: Optional[List[str]] = None,
    verbose: bool = True,
) -> Dict:
    """
    Executa o fluxo completo: análise (1+2), invocação dos N agentes (3),
    payload avaliador (4), avaliador (5), entrega (6).
    Retorna dict com analise, respostas_agentes, avaliacao, entrega_final.
    """
    model = model or os.environ.get("MODELO_DEFAULT")
    skills = skills_list if skills_list is not None else skills_loaded
    agentes = agentes

    if verbose:
        print("[MAESTRO] Iniciando fluxo. Pergunta:", (pergunta[:80] + "...") if len(pergunta) > 80 else pergunta)

    # --- Passo 1+2: Análise da pergunta ---
    if verbose:
        print("[MAESTRO] Passo 1+2: Analisando pergunta (chamando LLM Maestro)...")
    skill_maestro = get_skill_by_id(skills, "maestro")
    msg_analise = (
        pergunta
        + "\n\nRetorne um JSON com uma única chave 'analise' (objeto) contendo: "
        "pergunta (string), dominios_identificados (lista de strings), tipo_resposta_esperada (uma de: factual, analítica, técnica, criativa, comparativa), "
        "complexidade (uma de: baixa, média, alta), informacao_central (string)."
    )
    resp_analise = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": skill_maestro["content"]},
            {"role": "user", "content": msg_analise},
        ],
        response_format={"type": "json_object"},
    )
    raw_analise = (resp_analise.choices[0].message.content or "").strip()
    analise_data = extrair_json(raw_analise)
    analise = (analise_data.get("analise") or analise_data) if isinstance(analise_data, dict) else {}
    if not analise:
        analise = {"pergunta": pergunta, "dominios_identificados": [], "tipo_resposta_esperada": "analítica", "complexidade": "média", "informacao_central": pergunta}
    contexto_maestro = f"Domínios: {analise.get('dominios_identificados', [])}; Tipo: {analise.get('tipo_resposta_esperada', '')}; Complexidade: {analise.get('complexidade', '')}; Central: {analise.get('informacao_central', '')}"
    if verbose:
        print("[MAESTRO] Análise obtida:", analise.get("dominios_identificados"), analise.get("tipo_resposta_esperada"), analise.get("complexidade"))

    # --- Passo 3: Invocar os agentes ---
    respostas_agentes = []
    for skill_id in agentes:
        if verbose:
            print("[MAESTRO] Passo 3: Invocando agente:", skill_id)
        payload_maestro = {
            "skill_invocada": skill_id,
            "pergunta": pergunta,
            "contexto_maestro": contexto_maestro,
            "tipo_resposta_esperada": analise.get("tipo_resposta_esperada", "analítica"),
            "instrucao": "Responda estritamente dentro do seu domínio. Calcule seus scores.",
        }
        raw = invocar_agente_maestro(client, skill_id, payload_maestro, model=model, skills_list=skills)
        parsed = extrair_json(raw)
        if parsed and isinstance(parsed, dict):
            parsed.setdefault("agente_id", skill_id)
            respostas_agentes.append(parsed)
            if verbose:
                print("[MAESTRO]   ->", skill_id, "pode_responder:", parsed.get("pode_responder"), "score_final:", (parsed.get("scores") or {}).get("score_final"))
        else:
            respostas_agentes.append({"agente_id": skill_id, "agente_nome": skill_id, "pode_responder": False, "justificativa_viabilidade": "Resposta não veio em JSON válido.", "resposta": "", "scores": {}})
            if verbose:
                print("[MAESTRO]   ->", skill_id, "pode_responder: False (parse falhou)")

    # --- Passo 4: Montar payload do avaliador ---
    para_avaliador = [r for r in respostas_agentes if r.get("pode_responder") is True]
    if verbose:
        print("[MAESTRO] Passo 4: Respostas a enviar ao avaliador:", len(para_avaliador), "de", len(respostas_agentes))

    avaliacao = None
    if not para_avaliador:
        entrega_final = "## Resposta do Maestro\n\n**Pergunta:** " + pergunta + "\n\nNenhum agente qualificado para responder. Sugestão: reformule a pergunta ou consulte um domínio mais específico."
    else:
        # --- Passo 5: Invocar avaliador de coerência ---
        if verbose:
            print("[MAESTRO] Passo 5: Invocando avaliador de coerência...")
        payload_aval = {
            "pergunta_original": pergunta,
            "tipo_resposta_esperada": analise.get("tipo_resposta_esperada", "analítica"),
            "respostas_coletadas": [
                {
                    "agente_id": r.get("agente_id"),
                    "agente_nome": r.get("agente_nome", r.get("agente_id")),
                    "resposta": r.get("resposta", ""),
                    "scores_agente": r.get("scores") or {},
                    "limitacoes_da_resposta": r.get("limitacoes_da_resposta", ""),
                }
                for r in para_avaliador
            ],
        }
        skill_aval = get_skill_by_id(skills, "avaliador-coerencia")
        user_aval = json.dumps(payload_aval, ensure_ascii=False) + "\n\nRetorne APENAS um JSON com: avaliacao_completa (lista de objetos com agente_id, agente_nome, scores_avaliador, score_total, status, observacoes), ranking_final (lista de agente_id), conflitos_detectados, respostas_excluidas, threshold_utilizado."
        resp_aval = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": skill_aval["content"]},
                {"role": "user", "content": user_aval},
            ],
            response_format={"type": "json_object"},
        )
        raw_aval = (resp_aval.choices[0].message.content or "").strip()
        avaliacao = extrair_json(raw_aval)
        if verbose:
            ranking = (avaliacao or {}).get("ranking_final", [])
            print("[MAESTRO] Avaliador retornou ranking:", ranking)
            if (avaliacao or {}).get("conflitos_detectados"):
                print("[MAESTRO]   Conflitos:", avaliacao.get("conflitos_detectados"))

        # --- Passo 6: Entrega ao usuário ---
        if verbose:
            print("[MAESTRO] Passo 6: Formatando entrega ao usuário.")
        entrega_final = _formatar_entrega(pergunta, respostas_agentes, avaliacao, para_avaliador)

    if verbose:
        print("[MAESTRO] Fluxo concluído.")
    return {"analise": analise, "respostas_agentes": respostas_agentes, "avaliacao": avaliacao, "entrega_final": entrega_final}


def _formatar_entrega(pergunta: str, respostas_agentes: List[Dict], avaliacao: Optional[Dict], para_avaliador: List[Dict]) -> str:
    """Gera o markdown da entrega conforme PASSO 6 do Maestro."""
    n_consultados = len(respostas_agentes)
    aval = avaliacao or {}
    avaliacao_completa = aval.get("avaliacao_completa") or []
    ranking = aval.get("ranking_final") or [r.get("agente_id") for r in para_avaliador]
    # Ordenar por ranking_final (ordem de preferência)
    id_to_resp = {r.get("agente_id"): r for r in para_avaliador}
    ordenados = []
    for aid in ranking:
        if aid in id_to_resp:
            ordenados.append(id_to_resp[aid])
    for r in para_avaliador:
        if r.get("agente_id") not in ranking:
            ordenados.append(r)
    n_qualificadas = len(ordenados)
    linhas = [
        "## Resposta do Maestro",
        "",
        "**Pergunta:** " + pergunta,
        f"**Agentes consultados:** {n_consultados} | **Respostas qualificadas:** {n_qualificadas}",
        "---",
    ]
    for r in ordenados:
        aid = r.get("agente_id", "")
        nome = r.get("agente_nome", aid)
        score_total = None
        for item in avaliacao_completa:
            if item.get("agente_id") == aid:
                score_total = item.get("score_total")
                break
        if score_total is None:
            score_total = (r.get("scores") or {}).get("score_final", 0)
        pct = round((score_total or 0) * 100)
        linhas.append(f"### {nome} — {aid}")
        linhas.append(f"*Score de Confiança: {pct}%*")
        linhas.append("")
        linhas.append(r.get("resposta", ""))
        linhas.append("")
        linhas.append("---")
    linhas.append("")
    linhas.append("### Síntese")
    resumos = [(r.get("resposta", "")[:120] + "...") if len(r.get("resposta", "")) > 120 else r.get("resposta", "") for r in ordenados]
    linhas.append("Integração das perspectivas: " + " | ".join(resumos))
    return "\n".join(linhas)

In [ ]:
PERGUNTA_TESTE = (
    "Quais são as melhores práticas para uma startup de SaaS precificar o produto "
    "e quais ferramentas técnicas (APIs, billing) ajudam a implementar isso?"
)
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
model = os.environ.get("MODELO_DEFAULT")
resultado = executar_fluxo_maestro(client=client, pergunta=PERGUNTA_TESTE, model=model, agentes=["agente-tecnico", "agente-financeiro"], verbose=True)
print(resultado["entrega_final"])

[MAESTRO] Iniciando fluxo. Pergunta: Quais são as melhores práticas para uma startup de SaaS precificar o produto e q...
[MAESTRO] Passo 1+2: Analisando pergunta (chamando LLM Maestro)...
[MAESTRO] Análise obtida: ['Negócios / Estratégia de Precificação', 'Finanças (métricas e modelagem)', 'Técnico / Engenharia (APIs, billing, integrações)', 'Dados & Analytics (experimentos e métricas)', 'Mercado / Go-to-market (concorrência e posicionamento)'] analítica média
[MAESTRO] Passo 3: Invocando agente: agente-tecnico
[MAESTRO]   -> agente-tecnico pode_responder: True score_final: 0.9
[MAESTRO] Passo 3: Invocando agente: agente-financeiro
[MAESTRO]   -> agente-financeiro pode_responder: True score_final: 0.94
[MAESTRO] Passo 4: Respostas a enviar ao avaliador: 2 de 2
[MAESTRO] Passo 5: Invocando avaliador de coerência...
[MAESTRO] Avaliador retornou ranking: ['agente-financeiro', 'agente-tecnico']
[MAESTRO] Passo 6: Formatando entrega ao usuário.
[MAESTRO] Fluxo concluído.
## Resposta do Maes